# 0003 — MinerU as an OCR candidate

> **Superseded.** The run has been done — see `experiments/mineru/` for the raw
> output, the numbers, and the two gotchas this notebook does not encode:
> Colab's Python 3.13 has no vLLM wheels, and vLLM's spawned engine core needs
> the entry point guarded. Keep this only as a self-serve path.

**Question.** Can MinerU parse BRH's French PDFs well enough to be the OCR stage —
on a machine with no GPU of its own?

**Setup.** Google Colab, T4 GPU. MinerU 4.0 (`mineru.parser.parse`), tier
`standard` (small models + VLM), `ocr_mode="auto"` so born-digital pages are read
from the text layer and scans go through OCR.

**Input.** The six benchmark PDFs from `experiments/gold/documents/`, downloaded
from brh.ht by URL and verified by `sha256` — reproducible without uploading
anything.

**Cost.** Free tier. 27 pages take a few minutes after the first model download
(~2–3 GB, cached for the session).

> This notebook *is* the test — it has not been run on Colab from the repo.
> Run the smoke test before the full run; the install cell documents the
> fallbacks if the VLM engine or the import fails.


## 0 — Runtime

`Runtime → Change runtime type → T4 GPU`. Without a GPU, `standard` still runs
through llama.cpp on CPU, but it is slow; drop to `TIER = "basic"` below.


In [ ]:
import os, subprocess, sys

print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                      "--format=csv,noheader"], capture_output=True, text=True).stdout or
      "no GPU — set TIER = 'basic' below")
mem = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 2**30
print(f"system RAM: {mem:.1f} GiB  (standard wants ~16 GiB, T4 Colab has ~12.7)")
print("python:", sys.version.split()[0])


## 1 — Install

MinerU 4.0 renamed things: there is no `magic-pdf`, no `-b pipeline`, and the
extras are `torch` / `full` rather than `core` / `pipeline`.

| `EXTRA` | engines | when |
|---|---|---|
| `full` | ONNX + PyTorch + llama.cpp + vLLM | **recommended on a T4** — fastest `standard` |
| `torch` | ONNX + PyTorch + llama.cpp | lighter, if vLLM will not build or OOMs |
| `""` | ONNX + llama.cpp (base) | CPU-only, or to check the package quickly |

`standard` = small models **+ VLM**. `basic` = small models only (no VLM), and is
the fallback if VRAM or RAM is too tight.


In [ ]:
import subprocess, sys, importlib

EXTRA = "full"                      # "full" | "torch" | ""
spec = f"mineru[{EXTRA}]>=4.0,<5" if EXTRA else "mineru>=4.0,<5"
print("installing", spec)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", spec], check=True)

importlib.invalidate_caches()
try:
    import mineru
    print("mineru imported:", getattr(mineru, "__version__", "ok"))
except Exception as exc:
    print(f"import failed: {type(exc).__name__}: {exc}")
    print("Do Runtime -> Restart session, then re-run cells 0 and 2 onward. pip work is done.")


In [ ]:
!mineru-kit --version 2>&1 | tail -2


## 2 — The documents

Downloaded by URL and checked against the `sha256` in the filename: the same six
files the reference was built from, byte for byte. If brh.ht blocks the download,
the fallback cell uploads a zip of `experiments/gold/documents/` instead.


In [ ]:
import hashlib, pathlib, urllib.request

DOCS = pathlib.Path("/content/documents")
DOCS.mkdir(exist_ok=True)

PDFS = {
    "110b25ca611ddf3dd50eda474d60e4ba5c427c3d3978ed0a14292d8f926ee349": "https://www.brh.ht/wp-content/uploads/NOTE-MENSUELLE-DINFLATION-DU-MOIS-DE-NOVEMBRE-2025.pdf",
    "1a84a9a73493ae39e02993fbffddb76baee572cfa957193b80b51c8dcd78280a": "https://www.brh.ht/wp-content/uploads/2018/08/lc_02.pdf",
    "63b4ca02772a885f4fcf3a68ed6264af1b8bf2fdac25a56a0d1b08ac05d84451": "https://www.brh.ht/wp-content/uploads/Note-mensuelle-dinflation-septembre-2022_f.pdf",
    "6ab9508ca873d1055feb678003fe0fe56e6ecc518bdb48793fbd86d1537c7d67": "https://www.brh.ht/wp-content/uploads/Circulaire-122-pdf.pdf",
    "f2d799b9527eef9caaede5f0c5ee0afe4eb897ff8de972560f7b6e47e58faef9": "https://www.brh.ht/wp-content/uploads/2018/08/circulaire_95-1a.pdf",
    "f31dfee9575ff1197dc5ff28cc4d42b85d79234f9c58cb2f12a202768bc4ae01": "https://www.brh.ht/wp-content/uploads/pibsecteur.pdf",
}

def download(url, path):
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0 (BRH-Intelligence benchmark)"})
    with urllib.request.urlopen(req, timeout=90) as response:
        data = response.read()
    path.write_bytes(data)
    return data

for sha, url in PDFS.items():
    path = DOCS / f"{sha}.pdf"
    if not path.is_file():
        try:
            download(url, path)
        except Exception as exc:
            print(f"{sha[:8]}  download failed: {type(exc).__name__}: {exc}")
            continue
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    print(f"{sha[:8]}  {'ok' if digest == sha else 'SHA MISMATCH'}  {path.stat().st_size/1024:.0f} kB")

print(f"\n{len(list(DOCS.glob('*.pdf')))}/6 PDFs in {DOCS}")


**Fallback** if downloads fail: run the cell below, upload a zip of
`experiments/gold/documents/`, and skip the download cell.


In [ ]:
# from google.colab import files
# import zipfile
# up = files.upload()
# for name in up:
#     zipfile.ZipFile(name).extractall(DOCS)
# print(sorted(p.name[:8] for p in DOCS.glob('*.pdf')))


## 3 — Settings, and a smoke test on the hardest page

The settings that matter:

- **`tier="standard"`** — small models + VLM. `advanced` spends more compute on
  difficult pages and is the better choice once the pipeline works.
- **`ocr_mode="auto"`** — use the text layer where there is one, OCR where there
  is not. `txt` never falls back to OCR (scans come out empty); `ocr` re-reads
  born-digital pages it did not need to.
- **Language** — there is **no `fr`**: MinerU's `ch` model covers *Latin* script,
  and `-l/--lang` only applies to the legacy `pipeline` backend, not to `standard`.
  Passing `fr` raises `ValueError`. Leave the default.

Page 9 of the 2026 inflation note is a two-column glossary — the page every other
approach has fumbled, so it is the right smoke test.


In [ ]:
import json, pathlib, time
from mineru.parser import parse

TIER = "standard"     # flash | basic | standard | advanced
OCR_MODE = "auto"     # auto | txt | ocr

SHA = "110b25ca611ddf3dd50eda474d60e4ba5c427c3d3978ed0a14292d8f926ee349"

t0 = time.time()
result = parse(DOCS / f"{SHA}.pdf", tier=TIER, ocr_mode=OCR_MODE, page_range="9")
markdown = result.markdown()
print(f"{time.time() - t0:.1f}s  {len(markdown):,} chars\n")
print(markdown[:2500])


In [ ]:
# The result shape is worth seeing once: it decides how pages are split later.
print(type(result))
print([a for a in dir(result) if not a.startswith("_")])
try:
    js = json.loads(result.to_json())
    print("\ntop-level keys:", list(js)[:20])
    print("page-like entry:", json.dumps(js.get("pages", js)[0], ensure_ascii=False)[:300] if js.get("pages") else "-")
except Exception as exc:
    print("to_json:", type(exc).__name__, exc)


## 4 — Full run

All pages of all six documents. Failures are caught per file so one bad PDF does
not lose the run.


In [ ]:
OUT = pathlib.Path("/content/mineru_out")
OUT.mkdir(exist_ok=True)
rows = []

for sha in PDFS:
    pdf = DOCS / f"{sha}.pdf"
    if not pdf.is_file():
        continue
    t0 = time.time()
    try:
        result = parse(pdf, tier=TIER, ocr_mode=OCR_MODE)   # all pages
        markdown = result.markdown()
        payload = result.to_json()
        (OUT / f"{sha}.md").write_text(markdown, encoding="utf-8")
        (OUT / f"{sha}.json").write_text(payload, encoding="utf-8")
        try:
            pages = len(json.loads(payload).get("pages") or [])
        except Exception:
            pages = None
        elapsed = round(time.time() - t0, 1)
        rows.append({"sha": sha, "ok": True, "pages": pages, "chars": len(markdown), "seconds": elapsed})
        print(f"{sha[:8]}  ok   {len(markdown):>7,} chars  {str(pages):>3} pages  {elapsed:>6.1f}s")
    except Exception as exc:
        import traceback; traceback.print_exc()
        rows.append({"sha": sha, "ok": False, "error": f"{type(exc).__name__}: {exc}"})
        print(f"{sha[:8]}  FAILED  {type(exc).__name__}: {exc}")

(OUT / "results.json").write_text(
    json.dumps({"tier": TIER, "ocr_mode": OCR_MODE, "rows": rows}, indent=1, ensure_ascii=False),
    encoding="utf-8",
)
print(f"\nwritten to {OUT}")


## 5 — What came back

Characters per document, time per document, and the total. Compare against the
reference: **39 323 characters over 27 pages**, and the reference's own parse cost
was 390 LlamaParse credits.


In [ ]:
import pandas as pd

frame = pd.DataFrame(rows)
if "chars" in frame:
    frame["chars_per_s"] = (frame["chars"] / frame["seconds"]).round(0)
display(frame)

ok = frame[frame["ok"] == True] if "ok" in frame else frame
if len(ok):
    print(f"{len(ok)}/{len(frame)} documents, {ok['chars'].sum():,} chars, "
          f"{ok['seconds'].sum()/60:.1f} min, {ok['pages'].sum() if 'pages' in ok else '?'} pages")


In [ ]:
# A glance at the two hardest things: the glossary page and a wide table.
for sha, name in (( "110b25ca611ddf3dd50eda474d60e4ba5c427c3d3978ed0a14292d8f926ee349", "glossary p9"),
                  ("f31dfee9575ff1197dc5ff28cc4d42b85d79234f9c58cb2f12a202768bc4ae01", "wide financial table")):
    path = OUT / f"{sha}.md"
    if path.is_file():
        text = path.read_text(encoding="utf-8")
        marker = "GLOSSAIRE" if "glossary" in name else "Produit intérieur"
        at = text.find(marker)
        print(f"\n===== {name} =====\n" + (text[at:at + 1600] if at >= 0 else text[:1600]))


## 6 — Bring the output home

Download the zip, or mount Drive and copy it there. Drop the `.md` files into
`experiments/mineru/` in the repo (or attach them to the issue/PR) and the same
scoring that produced the reference can run on them.


In [ ]:
import shutil

archive = shutil.make_archive("/content/mineru_out", "zip", OUT)
print(archive, f"{pathlib.Path(archive).stat().st_size/1024:.0f} kB")

# from google.colab import drive, files
# drive.mount("/content/drive")
# shutil.copy(archive, "/content/drive/MyDrive/mineru_out.zip")
# files.download(archive)
